In [1]:
!pip install transformers datasets torch accelerate sentencepiece gradio

In [2]:
import json

with open("dataset.json", "r") as f:
    data = json.load(f)

print(data[:2])

[{'question': 'What is RCOEM?', 'answer': 'RCOEM is Shri Ramdeobaba College of Engineering and Management located in Nagpur.'}, {'question': 'Where is RCOEM located?', 'answer': 'RCOEM is located in Nagpur, Maharashtra.'}]


In [3]:
inputs = []
targets = []

for item in data:
    inputs.append("question: " + item["question"])
    targets.append(item["answer"])

print(inputs[0])
print(targets[0])

question: What is RCOEM?
RCOEM is Shri Ramdeobaba College of Engineering and Management located in Nagpur.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
encodings = tokenizer(
    inputs,
    truncation=True,
    padding=True,
    max_length=128
)

labels = tokenizer(
    targets,
    truncation=True,
    padding=True,
    max_length=128
)

encodings["labels"] = labels["input_ids"]

In [6]:
import torch

class ChatDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

    def __len__(self):
        return len(self.encodings["input_ids"])

dataset = ChatDataset(encodings)

In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_steps=5
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,10.505736
2,10.064542
3,10.365464
4,10.320402
5,10.229570
6,10.306437
7,10.300698
8,9.939271
9,10.176006
10,10.230649


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
model.save_pretrained("rcoem-chatbot")
tokenizer.save_pretrained("rcoem-chatbot")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('rcoem-chatbot/tokenizer_config.json', 'rcoem-chatbot/tokenizer.json')

In [10]:
def ask_question(question):
    input_text = "question: " + question

    inputs = tokenizer(input_text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_length=50
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

print(ask_question("Does RCOEM provide hostel facilities?"))

RCOEM provides hostel facilities.


In [ ]:
import gradio as gr
import json

# Load dataset
with open("dataset.json", "r") as f:
    data = json.load(f)

# Chatbot Function
def chatbot(message, history):

    message = message.lower().strip()

    best_match = None
    best_score = 0

    # Search through dataset
    for item in data:

        text = (
            item["question"].lower() + " " +
            item["answer"].lower()
        )

        score = 0

        # Count matching words
        for word in message.split():

            if word in text:
                score += 1

        # Save best match
        if score > best_score:
            best_score = score
            best_match = item

    # Return answer
    if best_match and best_score > 0:
        return best_match["answer"]

    return "Sorry, I don't know the answer yet."

# Gradio Chat Interface
demo = gr.ChatInterface(
    fn=chatbot,
    title="RCOEM AI ChatBot",
    description="Ask questions related to RCOEM",
    chatbot=gr.Chatbot(height=500)
)

# Launch App with Public Link
demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_1419/117737718.py:48: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=500)
/tmp/ipykernel_1419/117737718.py:48: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=500)
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5ed1c4e6f01badb8da.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
